# Project - Electricity Demand Forecasting with Google TimesFM

This notebook uses **TimesFM 2.0 (500M)** by Google Research to forecast electricity demand. Same dataset and train/test split as the XGBoost baseline, with a direct comparison at the end.

## 1. Import Dependencies

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import torch
from timesfm import TimesFmHparams, TimesFmCheckpoint, TimesFm
from sklearn.metrics import mean_squared_error, mean_absolute_error
print("Torch version:", torch.__version__)

## 2. Load Data from DuckDB

In [ ]:
DB_PATH = r"C:\Users\tusha\Desktop\ML\project_elec_demand\electricity_data.db"
con = duckdb.connect(DB_PATH)
data = con.execute("""
    SELECT
        timestamp AS Timestamp,
        hour, dayofweek, month, year, dayofyear,
        temperature AS Temperature,
        humidity AS Humidity,
        demand AS Demand
    FROM demand_records
    ORDER BY timestamp
""").fetchdf()
con.close()
print(f"Loaded {len(data)} rows")
data.head()

## 3. Feature Engineering (same as XGBoost pipeline)

In [ ]:
data["Timestamp"] = pd.to_datetime(data["Timestamp"])
data = data.set_index("Timestamp")
data["quarter"] = data.index.quarter
data["weekofyear"] = data.index.isocalendar().week.astype(int)
data["is_weekend"] = data.index.dayofweek.isin([5, 6]).astype(int)
print(f"Shape after feature engineering: {data.shape}")
data.head(3)

## 4. Train / Test Split (same boundary)

In [ ]:
train = data.loc[: "2023-12-31"].copy()
test  = data.loc["2024-01-01":].copy()
print(f"Train: {len(train)} rows  ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Test:  {len(test)} rows   ({test.index[0].date()} to {test.index[-1].date()})")

## 5. Load TimesFM 2.0 (500M) Model

TimesFM 2.0 supports up to 2048 context length. We fix the horizon at 128 and use a **rolling window** to cover the full test period.

In [ ]:
print("Loading TimesFM 2.0 (500M) model from HuggingFace...")
hparams = TimesFmHparams(
    context_len=2048,
    horizon_len=128,
    input_patch_len=32,
    output_patch_len=128,
    num_layers=50,
    model_dims=1280,
    per_core_batch_size=32,
    backend="cpu",
    use_positional_embedding=False,
)
checkpoint = TimesFmCheckpoint(
    huggingface_repo_id="google/timesfm-2.0-500m-pytorch",
)
tfm = TimesFm(hparams=hparams, checkpoint=checkpoint)
print("Model loaded successfully.")

## 6. Rolling-Window Forecast

Since TimesFM’s horizon is 128 per call, we step through the test set in blocks, always providing the **last available demand values** as context.

In [ ]:
context_len = 2048
horizon = 128
all_demand = pd.concat([train["Demand"], test["Demand"]]).values
n_train = len(train)
n_test  = len(test)
timesfm_preds = np.full(n_test, np.nan)
n_windows = (n_test + horizon - 1) // horizon
for start in range(0, n_test, horizon):
    end = min(start + horizon, n_test)
    ctx_end = n_train + start
    ctx_start = max(0, ctx_end - context_len)
    ctx = all_demand[ctx_start:ctx_end]
    try:
        point, _ = tfm.forecast(inputs=[ctx], freq=[0])
        # Take only what we need (forecast may produce up to horizon_len=128)
        h = min(end - start, point.shape[1])
        timesfm_preds[start:start+h] = point[0, :h]
    except Exception as e:
        print(f"  Window {start//horizon+1} failed: {e}")
    if (start // horizon) % 5 == 0:
        done = np.sum(~np.isnan(timesfm_preds))
        print(f"  Window {start//horizon+1}/{n_windows}: predicted {done}/{n_test}")
# Fill any remaining NaN with forward fill
nan_count = np.sum(np.isnan(timesfm_preds))
if nan_count > 0:
    print(f"WARNING: {nan_count} NaN predictions, filling with forward fill")
    mask = np.isnan(timesfm_preds)
    idx = np.where(~mask, np.arange(n_test), 0)
    np.maximum.accumulate(idx, axis=0, out=idx)
    timesfm_preds[mask] = timesfm_preds[idx[mask]]
print(f"\nGenerated {np.sum(~np.isnan(timesfm_preds))} predictions")


## 7. Evaluate TimesFM Performance

In [ ]:
y_true = test["Demand"].values
y_pred = timesfm_preds
rmse_tfm = np.sqrt(mean_squared_error(y_true, y_pred))
mae_tfm  = mean_absolute_error(y_true, y_pred)
mape_tfm = (np.abs(y_true - y_pred) / y_true).mean() * 100
print(f"TimesFM  RMSE: {rmse_tfm:.2f}")
print(f"TimesFM  MAE:  {mae_tfm:.2f}")
print(f"TimesFM  MAPE: {mape_tfm:.2f} %")

## 8. Compare with XGBoost Baseline

In [ ]:
con = duckdb.connect(r"C:\Users\tusha\Desktop\ML\project_elec_demand\model_reports.db")
xgb_row = con.execute("SELECT rmse, mae FROM model_results WHERE model_name = 'XGBoost'").fetchone()
con.close()
xgb_rmse, xgb_mae = xgb_row
comparison = pd.DataFrame({
    "Model": ["TimesFM 2.0 (500M)", "XGBoost"],
    "RMSE":  [f"{rmse_tfm:.2f}", f"{xgb_rmse:.2f}"],
    "MAE":   [f"{mae_tfm:.2f}", f"{xgb_mae:.2f}"],
    "MAPE":  [f"{mape_tfm:.2f}%", "2.57%"],
})
print("\n========== MODEL COMPARISON ==========")
print(comparison.to_string(index=False))

## 9. Visualize Predictions

In [ ]:
plt.figure(figsize=(16, 6))
plt.plot(test.index, y_true, label="Actual Demand", color="blue", alpha=0.8)
plt.plot(test.index, y_pred, label="TimesFM Forecast", color="red", linestyle="--", alpha=0.7)
plt.title("Electricity Demand Forecasting — TimesFM 2.0 vs Actual")
plt.xlabel("Date")
plt.ylabel("Demand (MW)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 6))
errors = y_true - y_pred
plt.plot(test.index, errors, color="purple", alpha=0.6)
plt.axhline(y=0, color="black", linestyle="-", linewidth=0.5)
plt.title("Forecast Errors (Actual − Predicted)")
plt.xlabel("Date")
plt.ylabel("Error (MW)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Tabular Report

In [ ]:
# ====== TIMESFM MODEL PERFORMANCE REPORT ======

# --- 1. Performance Summary ---
metrics_df = pd.DataFrame({
    "Metric": ["RMSE", "MAE", "MAPE (%)", "Test Samples"],
    "TimesFM 2.0": [f"{rmse_tfm:.2f}", f"{mae_tfm:.2f}", f"{mape_tfm:.2f}", f"{n_test}"],
    "XGBoost":     [f"{xgb_rmse:.2f}", f"{xgb_mae:.2f}", "2.57%", f"{n_test}"],
})
print("=" * 70)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 70)
print(metrics_df.to_string(index=False))
print()

# --- 2. Actual vs Predicted Sample ---
sample_n = min(20, n_test)
compare_df = pd.DataFrame({
    "Timestamp": test.index[:sample_n],
    "Actual": y_true[:sample_n],
    "TimesFM_Pred": y_pred[:sample_n],
    "Error": y_true[:sample_n] - y_pred[:sample_n],
    "Error_%": (y_true[:sample_n] - y_pred[:sample_n]) / y_true[:sample_n] * 100,
})
for c in ["Actual", "TimesFM_Pred", "Error"]:
    compare_df[c] = compare_df[c].apply(lambda x: f"{x:.2f}")
compare_df["Error_%"] = compare_df["Error_%"].apply(lambda x: f"{x:.2f}%")
print("=" * 110)
print("ACTUAL vs PREDICTED (First 20 Test Samples)")
print("=" * 110)
print(compare_df.to_string(index=False))
print()

# --- 3. Summary Statistics ---
over_pred = np.sum(y_pred > y_true)
under_pred = np.sum(y_pred < y_true)
summary_df = pd.DataFrame({
    "Statistic": [
        "Mean Actual", "Mean Predicted",
        "Std Actual", "Std Predicted",
        "Min Actual", "Max Actual",
        "Min Predicted", "Max Predicted",
        "Over-Predictions", "Under-Predictions",
    ],
    "Value": [
        f"{y_true.mean():.2f}", f"{y_pred.mean():.2f}",
        f"{y_true.std():.2f}", f"{y_pred.std():.2f}",
        f"{y_true.min():.2f}", f"{y_true.max():.2f}",
        f"{y_pred.min():.2f}", f"{y_pred.max():.2f}",
        over_pred, under_pred,
    ]
})
print("=" * 50)
print("PREDICTION SUMMARY STATISTICS")
print("=" * 50)
print(summary_df.to_string(index=False))
print()

# --- 4. Persist to model_reports.db ---
con = duckdb.connect(r"C:\Users\tusha\Desktop\ML\project_elec_demand\model_reports.db")
con.execute("INSERT INTO model_results (model_name, rmse, mae) VALUES (?, ?, ?)",
            ["TimesFM_2.0", float(f"{rmse_tfm:.2f}"), float(f"{mae_tfm:.2f}")])
preds_to_insert = [(ts, float(a), float(p), float(a-p), float(abs(a-p)/a*100))
                   for ts, a, p in zip(test.index[:100], y_true[:100], y_pred[:100])]
con.executemany("INSERT INTO predictions (timestamp, actual, predicted, error, mape) VALUES (?, ?, ?, ?, ?)",
                preds_to_insert)
con.close()
print(f"Inserted metrics + 100 predictions into model_reports.db")
print()
print("=" * 50)
print("END OF TIMESFM REPORT")
print("=" * 50)

---